In [2]:
import torch
import torch.nn as nn

class MultilayerLSTM(nn.Module):
    def __init__(self, n_features=40, hidden_dim1=64, hidden_dim2=32, dense_dim=16, dropout_prob=0.2):
        super(MultilayerLSTM, self).__init__()
        
        # 1. Warstwa LSTM (odpowiednik return_sequences=True)
        # Wejście: (batch_size, seq_len, 40) -> Wyjście: (batch_size, seq_len, 64)
        self.lstm1 = nn.LSTM(input_size=n_features, hidden_size=hidden_dim1, batch_first=True)
        self.dropout1 = nn.Dropout(dropout_prob)
        
        # 2. Warstwa LSTM
        # Wejście: (batch_size, seq_len, 64) -> Wyjście: (batch_size, seq_len, 32)
        self.lstm2 = nn.LSTM(input_size=hidden_dim1, hidden_size=hidden_dim2, batch_first=True)
        self.dropout2 = nn.Dropout(dropout_prob)
        
        # Warstwy w pełni połączone (Dense)
        self.fc1 = nn.Linear(hidden_dim2, dense_dim)
        self.relu = nn.ReLU()
        self.fc_out = nn.Linear(dense_dim, 1)  # Wyjście: 1 wartość (regresja)

    def forward(self, x):
        # x shape: (batch_size, seq_len, n_features)
        
        # 1. Przejście przez pierwszy LSTM
        out, _ = self.lstm1(x)
        out = self.dropout1(out)
        
        # 2. Przejście przez drugi LSTM
        out, _ = self.lstm2(out)
        
        # Odpowiednik return_sequences=False:
        # Bierzemy tylko wektor z ostatniego kroku czasowego (indeks -1)
        out = out[:, -1, :]  # shape: (batch_size, hidden_dim2)
        out = self.dropout2(out)
        
        # Warstwy gęste
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc_out(out)  # shape: (batch_size, 1)
        
        return out

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 3. Inicjalizacja architektury i załadowanie wag
N_FEATURES = 39  # Liczba Twoich cech
model = LSTMRegressor(input_size=N_FEATURES).to(device)

model.load_state_dict(torch.load('model_lstm.pth', map_location=device))

# KRUCJALNE: Przełączenie w tryb ewaluacji (wyłącza Dropout na czas wnioskowania)
model.eval()

# 4. Wczytanie scalera (opcjonalnie, jeśli dane wymagają przeskalowania)
# scaler = joblib.load('scaler.joblib')

# 5. Przygotowanie danych wejściowych
# Model oczekuje sekwencji o kształcie: (batch_size, sequence_length, n_features)
# W Twoim przypadku: (1, 24, 39) -> 1 próbka, 24 godziny historii, 39 cech

# Przykład dla nowej macierzy danych o wymiarach 24 x 39:
# raw_data = np.array(...) # Twoje nowe dane dla 24 godzin
# scaled_data = scaler.transform(raw_data) # Opcjonalne skalowanie

dummy_input = np.random.randn(24, N_FEATURES)  # Atrapa 24 godzin danych

# Konwersja na Tensor i dodanie wymiaru batcha (1, 24, 39)
input_tensor = torch.tensor(dummy_input, dtype=torch.float32).unsqueeze(0).to(device)

# 6. Predykcja bez obliczania gradientów
with torch.no_grad():
    prediction = model(input_tensor)

# Pobranie wyniku jako zwykła liczba float
output_kwh = prediction.item()
print(f"Przewidywane zużycie energii (total_kwh): {output_kwh:.4f}")